In [27]:
import numpy as np
import pandas as pd

In [28]:
df = pd.read_csv("../Data/spam.csv" , encoding='latin-1')
df = df[['v1' , 'v2']]
df.columns = ['label', 'message' ]
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [29]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["message"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [30]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(binary=True)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

In [31]:
class BernoulliNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def _binarize(self, X):
        from scipy.sparse import issparse
        X = X.toarray() if issparse(X) else np.asarray(X)
        return (X > 0).astype(int)
    
    def fit(self, X, y):
        X = self._binarize(X) 
        y = np.asarray(y)
        self.classes = np.unique(y)

        n_features = X.shape[1]
        self.log_prior = np.zeros(len(self.classes))
        self.p_present = np.zeros((len(self.classes), n_features))

        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            self.log_prior[i] = np.log(len(X_c) / len(X))
            docs_with_word = X_c.sum(axis=0)
            self.p_present[i] = (docs_with_word + self.alpha) / (len(X_c) + 2 * self.alpha)

        return self

    def predict(self, X):
        X = (X.toarray() > 0).astype(int)
        log_p = np.log(self.p_present)
        log_not_p = np.log(1 - self.p_present)
        scores = X @ log_p.T + (1 - X) @ log_not_p.T + self.log_prior

        return self.classes[np.argmax(scores, axis=1)]

In [32]:
my_bernoulli = BernoulliNB(alpha=1)

my_bernoulli.fit(
    X_train,
    y_train
)

y_pred_scratch = my_bernoulli.predict(X_test)

In [33]:
from sklearn.naive_bayes import BernoulliNB

sk_nb = BernoulliNB(
    alpha=1,
    binarize=None
)

sk_nb.fit(
    X_train,
    y_train
)

y_pred_sklearn = sk_nb.predict(X_test)

In [34]:
from sklearn.metrics import accuracy_score

print(
    "Scratch:",
    accuracy_score(y_test, y_pred_scratch)
)

print(
    "Sklearn:",
    accuracy_score(y_test, y_pred_sklearn)
)

Scratch: 0.9757847533632287
Sklearn: 0.9757847533632287


In [35]:
from sklearn.metrics import confusion_matrix

cm1 = confusion_matrix(y_test, y_pred_scratch)
print(cm1)

cm2 = confusion_matrix(y_test, y_pred_sklearn)
print(cm2)

[[966   0]
 [ 27 122]]
[[966   0]
 [ 27 122]]
